# Notebook 03: Transfer Learning (MobileNetV2) & Test Evaluation

**Objectives:**
1. Understand Transfer Learning and why MobileNetV2 is ideal for mobile deployment.
2. **Phase A (Feature Extraction):** Frozen ImageNet backbone + custom head.
3. **Phase B (Fine-Tuning):** Unfreeze top layers with small learning rate ($10^{-5}$) and frozen BatchNorm.
4. Comprehensive Test set evaluation: Macro F1, Normalized Confusion Matrix, Top-3 Confused Class Pairs.

In [ ]:
# Cell 0: Setup Directories & Colab Dependencies
import os
!pip install -q importlib_resources tensorflow-datasets --upgrade

RESULTS_DIR = 'results' if os.path.exists('results') or not os.path.exists('../results') else '../results'
MODELS_DIR = 'models' if os.path.exists('models') or not os.path.exists('../models') else '../models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Results: {os.path.abspath(RESULTS_DIR)}")
print(f"Models:  {os.path.abspath(MODELS_DIR)}")

In [ ]:
# Cell 1: Environment & Seed
import random
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import time
from sklearn.metrics import classification_report, confusion_matrix, f1_score

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

In [ ]:
# Cell 2: Data Pipeline with MobileNetV2 Preprocessing
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

train_ds_raw, ds_info = tfds.load('plant_village', split='train[:70%]', as_supervised=True, with_info=True)
val_ds_raw = tfds.load('plant_village', split='train[70%:85%]', as_supervised=True)
test_ds_raw = tfds.load('plant_village', split='train[85%:]', as_supervised=True)

class_names = ds_info.features['label'].names
num_classes = ds_info.features['label'].num_classes

def preprocess_mobilenet(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = preprocess_input(image) # scales pixels to [-1, 1]
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds_raw.map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
train_ds = train_ds.shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE)

val_ds = val_ds_raw.map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)
test_ds = test_ds_raw.map(preprocess_mobilenet, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("MobileNetV2 datasets ready (GPU streaming mode).")

In [ ]:
# Cell 3: Build MobileNetV2 Model (Phase A: Frozen Backbone)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
x = tf.keras.layers.Dropout(0.2, seed=SEED, name='head_dropout')(x)
outputs = tf.keras.layers.Dense(num_classes, activation='softmax', name='classifier_head')(x)

mobilenet_model = tf.keras.Model(inputs, outputs, name='mobilenetv2_plantvillage')
mobilenet_model.summary()

In [ ]:
# Cell 4: Train Phase A (Feature Extraction)
mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

start_time = time.time()
history_phase_a = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=4,
    verbose=1
)
phase_a_time = time.time() - start_time
print(f"Phase A completed in {phase_a_time:.1f} seconds.")

In [ ]:
# Cell 5: Phase B (Fine-Tuning)
base_model.trainable = True

fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

finetuned_checkpoint = os.path.join(MODELS_DIR, 'mobilenetv2_finetuned.keras')
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint(finetuned_checkpoint, save_best_only=True)
]

start_time = time.time()
history_phase_b = mobilenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
    verbose=1
)
phase_b_time = time.time() - start_time
print(f"Phase B completed in {phase_b_time:.1f} seconds.")

In [ ]:
# Cell 6: Rigorous Evaluation on TEST SET ONLY
print("Evaluating fine-tuned MobileNetV2 on the hold-out Test Set...")
y_true = []
y_pred_probs = []

for images, labels in test_ds:
    preds = mobilenet_model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred = np.argmax(np.array(y_pred_probs), axis=1)

test_acc = np.mean(y_true == y_pred)
macro_f1 = f1_score(y_true, y_pred, average='macro')

print(f"\n" + "="*40)
print(f"TEST SET ACCURACY: {test_acc * 100:.2f}%")
print(f"TEST SET MACRO F1: {macro_f1 * 100:.2f}%")
print("="*40 + "\n")

report = classification_report(y_true, y_pred, target_names=class_names, digits=4)
print(report)
with open(os.path.join(RESULTS_DIR, 'classification_report.txt'), 'w') as f:
    f.write(report)

In [ ]:
# Cell 7: Normalized Confusion Matrix & Top-3 Confused Pairs
cm = confusion_matrix(y_true, y_pred)
cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(18, 15))
sns.heatmap(cm_normalized, xticklabels=class_names, yticklabels=class_names, cmap='Blues', annot=False)
plt.xlabel('Predicted Label', fontsize=12)
plt.ylabel('True Label', fontsize=12)
plt.title('Normalized Confusion Matrix on Hold-out Test Set', fontsize=14)
plt.xticks(rotation=90, fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'confusion_matrix.png'), dpi=200)
plt.show()

np.fill_diagonal(cm, 0)
confused_indices = np.unravel_index(np.argsort(cm.ravel())[::-1], cm.shape)

print("\nTop 3 Most Confused Class Pairs:")
for rank in range(3):
    true_idx = confused_indices[0][rank]
    pred_idx = confused_indices[1][rank]
    confused_count = cm[true_idx, pred_idx]
    print(f"{rank+1}. True: '{class_names[true_idx]}' misclassified as '{class_names[pred_idx]}' ({confused_count} times)")